In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime
from typing import List, Dict, Optional

#----------------------CONFIG----------------------
with open("/content/drive/MyDrive/Data Analysis/token.json", "r") as f:
    token = json.load(f)

GITHUB_TOKEN = token["TOKEN"]
REPO_OWNER = "django"
REPO_NAME = "django"
MAX_PAGES = 5
OUTPUT_FILE = "cycle_time.csv"

BASE_URL = "https://api.github.com"
HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}
#---------------------------------------------------


#----------------------HELPERS----------------------
def parse_date(date_str: str) -> datetime:
    return datetime.fromisoformat(date_str.replace("z", "+00:00"))


def safe_request(url: str, params: dict=None) -> dict:
    response = requests.get(url, headers=HEADERS, params=params)

    if response.status_code != 200:
        raise Exception(f"Request failed with status code {response.status_code}")

    return response.json()
#---------------------------------------------------


#------------------GITHUB SERVICE-------------------
def fetch_merged_pull_requests(rp_owner: str, rp_name: str, max_page: int) -> List[Dict]:
    all_pull_requests = []

    for page in range(1, max_page + 1):
        data = safe_request(
            f"{BASE_URL}/repos/{rp_owner}/{rp_name}/pulls",
            params={
                "state": "closed",
                "per_page": 100,
                "page": page,
                "sort": "merged",
                "direction": "desc"
            }
        )

        if not data:
            break

        merged_pull_requests = [pr for pr in data if pr["merged_at"]]
        all_pull_requests.extend(merged_pull_requests)

        if len(data) < 100:
            break
    return all_pull_requests


def fetch_first_commit_date(owner: str, name: str, pr_number: int) -> Optional[datetime]:
    data = safe_request(
        f"{BASE_URL}/repos/{owner}/{name}/pulls/{pr_number}/commits",
    )

    if not data:
        return None

    dates = [
        parse_date(c["commit"]["author"]["date"])
        for c in data
        if c.get("commit", {}).get("author", {}).get("date")
    ]

    return min(dates) if dates else None
#---------------------------------------------------

In [ ]:
#---------------------METRICS-----------------------
def calculate_cycle_time(start: datetime, end: datetime) -> float:
  return (end - start).total_seconds() / 3600
#---------------------------------------------------

def build_dateset(prs: List[Dict], rp_owner: str, rp_name: str) -> pd.DataFrame:
  rows = []

  for i, pr in enumerate(prs, 1):
    pr_number = pr["number"]

    create_at = parse_date(pr["created_at"])
    merged_at = parse_date(pr["merged_at"])

    first_commit = fetch_first_commit_date(rp_owner, rp_name, pr_number)
    cycle_time = (
        calculate_cycle_time(first_commit, merged_at)
        if first_commit else None
    )

    lead_time = calculate_cycle_time(create_at, merged_at)

    rows.append({
        "pr_number": pr_number,
        "author": pr["user"]["login"],
        "create_at": create_at,
        "merged_at": merged_at,
        "cycle_time_hours": round(cycle_time,2) if cycle_time else None,
        "lead_time_hours": round(lead_time,2)
    })

  return pd.DataFrame(rows)

In [ ]:
#---------------------MAIN-----------------------

def main():
  prs = fetch_merged_pull_requests(REPO_OWNER, REPO_NAME, MAX_PAGES)

  if not prs:
    print("No merged pull requests found.")
    return

  df = build_dateset(prs, REPO_OWNER, REPO_NAME)
  df.to_csv(OUTPUT_FILE, index=False, sep=";", decimal=",")
  print(f"Data saved to {OUTPUT_FILE}")

if __name__ == "__main__":
  main()

Data saved to cycle_time.csv
